# 02 — Core LOCO CV (acute-only)

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (p / "analysis" / "lib").exists())
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from joblib import Parallel, delayed

from analysis.lib.data import (load_label, restricted_pool,
                               build_demographics_matrix, identify_binary_targets)
from analysis.lib.pipelines import (process_fold, process_fold_majority,
                                    process_fold_permuted)
from analysis.lib.stacking import process_stacking_fold
from analysis.lib.cv import loco_cv

DATA_DIR   = PROJECT_ROOT / "data"
LACUNA_DIR = DATA_DIR / "lacuna" / "METAVCI_PSCI_LACUNA" / "derivatives"
OUT_DIR    = PROJECT_ROOT / "outputs" / "predictive_modeling_core" / "loco"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CLIN_PATH = DATA_DIR / "tabular" / "clinical_data.csv"
assert CLIN_PATH.exists(), f"missing {CLIN_PATH} — see data/README.md"

In [ ]:
clinical_df = pd.read_csv(CLIN_PATH, index_col=0)
acute_pool = restricted_pool(LACUNA_DIR, clinical_df, all_of=["acuteinfarct"])
clinical = clinical_df.loc[acute_pool]

lesion_acute = load_label(LACUNA_DIR, "acuteinfarct", "rd",  "damagepct").reindex(acute_pool)
flnm_acute   = load_label(LACUNA_DIR, "acuteinfarct", "fnm", "tmap").reindex(acute_pool)
slnm_acute   = load_label(LACUNA_DIR, "acuteinfarct", "snm", "disconnectionpct").reindex(acute_pool)
print(f"Pool: {len(acute_pool)} subjects")

X_demographics, demo_names, _, _, _ = build_demographics_matrix(clinical, exclude_cohort=True)
cohort_series = clinical["demographics-cohort"]
loco_splits = list(loco_cv(cohort_series))
print(f"LOCO splits: {len(loco_splits)} cohorts")

binary_targets = identify_binary_targets(clinical)
TARGETS_TO_RUN = binary_targets  # set to binary_targets[:1] to smoke-test


def splits_for(valid_mask):
    """Re-index LOCO splits to the valid-mask-restricted target population."""
    valid_idx = np.where(valid_mask.values)[0]
    mapping = {orig: i for i, orig in enumerate(valid_idx)}
    out = []
    for train, test in loco_splits:
        tr = np.array([mapping[o] for o in train if o in mapping], dtype=int)
        te = np.array([mapping[o] for o in test if o in mapping], dtype=int)
        if len(te) > 0 and len(tr) > 0:
            out.append((tr, te))
    return out

In [ ]:
# Single-view models + null models
results_list, all_scores_list = [], []

for target in TARGETS_TO_RUN:
    y_series = clinical[target]
    valid_mask = y_series.notna()
    y = y_series[valid_mask].values
    X_base = X_demographics[valid_mask.values]
    n_demo = X_base.shape[1]

    configs = {
        ("Baseline", "base"):   X_base,
        ("Acute",    "Lesion"): np.hstack([X_base, lesion_acute.loc[valid_mask].values]),
        ("Acute",    "fLNM"):   np.hstack([X_base, flnm_acute.loc[valid_mask].values]),
        ("Acute",    "sLNM"):   np.hstack([X_base, slnm_acute.loc[valid_mask].values]),
    }
    print(f"--- {target} (N={len(y)}) ---")
    target_summary = {"Target": target, "N": len(y)}

    for (model_name, modality), X in configs.items():
        fold_results = Parallel(n_jobs=-1)(
            delayed(process_fold)(i, train, test, X, y, model_name, target, n_demo)
            for i, (train, test) in enumerate(splits_for(valid_mask)))
        for r in fold_results:
            r["Modality"] = modality
        all_scores_list.extend(fold_results)
        key = f"{modality}_{model_name}" if modality != "base" else model_name
        target_summary[key] = np.nanmean([r["AUPRC"] for r in fold_results])
        target_summary[key + "_AUROC"] = np.nanmean([r.get("AUROC", np.nan) for r in fold_results])

    perm = Parallel(n_jobs=-1)(
        delayed(process_fold_permuted)(i, train, test, X_base, y, target, n_demo, rng_seed=42)
        for i, (train, test) in enumerate(splits_for(valid_mask)))
    for r in perm:
        r["Modality"] = "base"
    all_scores_list.extend(perm)
    target_summary["Permuted"] = np.nanmean([r["AUPRC"] for r in perm])
    target_summary["Permuted_AUROC"] = np.nanmean([r.get("AUROC", np.nan) for r in perm])

    maj = []
    for i, (train, test) in enumerate(splits_for(valid_mask)):
        r = process_fold_majority(i, train, test, y, target)
        r["Modality"] = "base"
        maj.append(r)
    all_scores_list.extend(maj)
    target_summary["Majority"] = np.nanmean([r["AUPRC"] for r in maj])
    target_summary["Majority_AUROC"] = np.nanmean([r.get("AUROC", np.nan) for r in maj])

    results_list.append(target_summary)

results_df = pd.DataFrame(results_list)
all_scores_df = pd.DataFrame(all_scores_list)
print(results_df.round(3))

In [ ]:
# 4-view stacker
stacking_scores_list, stacking_coefs_list = [], []

for target in TARGETS_TO_RUN:
    y_series = clinical[target]
    valid_mask = y_series.notna()
    y = y_series[valid_mask].values
    X_base = X_demographics[valid_mask.values]
    n_demo = X_base.shape[1]

    view_configs = [
        ("baseline",     X_base),
        ("lesion_acute", np.hstack([X_base, lesion_acute.loc[valid_mask].values])),
        ("flnm_acute",   np.hstack([X_base, flnm_acute.loc[valid_mask].values])),
        ("slnm_acute",   np.hstack([X_base, slnm_acute.loc[valid_mask].values])),
    ]
    print(f"--- Stacking: {target} (N={len(y)}) ---")
    fold_results = Parallel(n_jobs=-1)(
        delayed(process_stacking_fold)(i, train, test, view_configs, y, n_demo)
        for i, (train, test) in enumerate(splits_for(valid_mask)))
    for r in fold_results:
        stacking_scores_list.append({
            "Target": target, "Model": "Stacking", "Modality": "meta",
            "Fold": r["Fold"], "AUPRC": r["AUPRC"], "AUPRC_train": r["AUPRC_train"],
            "AUROC": r.get("AUROC", float("nan")),
            "AUROC_train": r.get("AUROC_train", float("nan")),
            "meta_C": r["meta_C"]})
        stacking_coefs_list.append({
            "Target": target, "Fold": r["Fold"],
            **{f"coef_{lbl}": c for lbl, c in zip(r["view_labels"], r["meta_coefs"])},
            "intercept": r["meta_intercept"]})
    print(f"  mean test AUPRC = {np.nanmean([r['AUPRC'] for r in fold_results]):.3f}")

stacking_scores_df = pd.DataFrame(stacking_scores_list)
stacking_coefs_df  = pd.DataFrame(stacking_coefs_list)

In [ ]:
results_df.to_csv(OUT_DIR / "modeling_summary.csv", index=False)
all_scores_df.to_csv(OUT_DIR / "all_scores.csv", index=False)
stacking_scores_df.to_csv(OUT_DIR / "stacking_scores.csv", index=False)
stacking_coefs_df.to_csv(OUT_DIR / "stacking_meta_coefficients.csv", index=False)

view_labels = ["baseline", "lesion_acute", "flnm_acute", "slnm_acute"]
coef_cols = [f"coef_{l}" for l in view_labels]
mean_coefs = stacking_coefs_df.groupby("Target")[coef_cols].mean().rename(
    columns={f"coef_{l}": l for l in view_labels})
mean_coefs.to_csv(OUT_DIR / "stacking_mean_coefficients.csv")
print(f"Saved to {OUT_DIR}")